# GAN-CNN — Kaggle GPU Training
Prepared for: GAN 50 epochs, CNN A+B 15 epochs (ResNet18), patience 5, synthetic 40/class.

**Before running:** Right sidebar -> Add Input -> add the *COVID-19 Radiography Database* dataset. Settings -> Accelerator -> **GPU T4 x2 or P100**. Internet ON (to clone GitHub).


In [ ]:
# 1. Check GPU - MUST say True and show a T4/P100
!nvidia-smi
import torch
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE - STOP, enable GPU in Settings')
print('PyTorch:', torch.__version__)


In [ ]:
# 2. Clone project (PyTorch is already on Kaggle - do NOT pip install torch)
!rm -rf Enhancing-Clinical-Data-with-GAN-Based-Synthetic-Data-Generation
!git clone https://github.com/nanthakishoren2005/Enhancing-Clinical-Data-with-GAN-Based-Synthetic-Data-Generation
%cd Enhancing-Clinical-Data-with-GAN-Based-Synthetic-Data-Generation
!pip install -q pyyaml seaborn
!ls


In [ ]:
# 3. Apply the recommended config (both CNNs identical = fair comparison)
import pathlib
p = pathlib.Path('configs/config.yaml')
t = p.read_text()
t = t.replace('  epochs: 8\n  save_sample_interval: 2', '  epochs: 50\n  save_sample_interval: 2')
t = t.replace('  epochs: 6\n  early_stopping_patience: 3', '  epochs: 15\n  early_stopping_patience: 5')
p.write_text(t)
print(p.read_text())


## Dataset
The prepare script expects `archive/COVID-19_Radiography_Dataset/Normal/images`, `.../COVID/images`, etc. The next cell finds that folder under `/kaggle/input` and links it.


In [ ]:
# 4. Find and link the Kaggle dataset
import os, pathlib, shutil
from pathlib import Path
candidates = list(Path('/kaggle/input').rglob('COVID-19_Radiography_Dataset')) + list(Path('/kaggle/input').rglob('Normal/images'))
print('Found:', candidates[:5])
# List top level so you can fix the path manually if not found
!ls -R /kaggle/input | head -100


In [ ]:
# 5. Link it as archive/ - EDIT src_root if the print above shows a different path
from pathlib import Path
import shutil
# Try auto-detect: folder that contains Normal/COVID/Lung_Opacity/Viral Pneumonia
src_root = None
for d in Path('/kaggle/input').rglob('COVID-19_Radiography_Dataset'):
    src_root = d; break
if src_root is None:
    # dataset uploaded flat: parent of Normal folder
    for d in Path('/kaggle/input').rglob('Normal'):
        if (d/'images').exists(): src_root = d.parent; break
print('Using:', src_root)
assert src_root is not None, 'Dataset not found - check cell 4 output and set src_root manually'
dest = Path('archive/COVID-19_Radiography_Dataset')
dest.parent.mkdir(exist_ok=True)
if dest.exists(): shutil.rmtree(dest)
shutil.copytree(src_root, dest)
print('Copied to', dest)


In [ ]:
# 6. Prepare splits: 600 train / 100 val / 100 test per class (from scripts/prepare_covid_dataset.py)
!python scripts/prepare_covid_dataset.py
!python -m src.data.validate_dataset


In [ ]:
# 7. Stage 1 - Train cDCGAN 50 epochs on GPU - 6-20 epochs is NOT enough for clear full chest X-rays. Check outputs/gan_samples every 10 epochs.
!python -m src.gan.train_gan --epochs 50 --batch-size 32


In [ ]:
# 8. Generate synthetic: 40 per class
!python -m src.gan.generate_images --num-per-class 40
!find data/synthetic -type f | wc -l


In [ ]:
# 9. Stage 2 - Model A Baseline, ResNet18, 15 epochs
!python -m src.cnn.train_baseline --epochs 15 --batch-size 32 --architecture resnet18


In [ ]:
# 10. Stage 3 - Model B GAN-Augmented, SAME settings as Model A
!python -m src.cnn.train_augmented --epochs 15 --batch-size 32 --architecture resnet18


In [ ]:
# 11. Compare on untouched real test set - this is your final report result
!python -m src.evaluation.compare_models
!cat outputs/reports/comparison_summary.csv


In [ ]:
# 12. Zip everything you need to download and run the dashboard locally
!zip -r /kaggle/working/gan_cnn_results.zip checkpoints outputs data/synthetic configs/config.yaml
print('Download gan_cnn_results.zip from Output, unzip into your local project, then: streamlit run app.py')
